In [0]:
data=spark.read.format('csv').option('inferSchema',True).option('header',True).load("/Volumes/sample_data/sales/bigmart/BigMart Sales.csv")

In [0]:
data.printSchema()

In [0]:
dbutils.fs.ls("/Volumes/sample_data/sales/bigmart")
data.show()
data.count()
data.columns
data.describe().show()
data.describe("Item_Outlet_Sales")

In [0]:
data.display()

In [0]:
df_json=spark.read.format('json').option('inferSchema',True).option('header',True).option('multiLine',False).load('/Volumes/sample_data/sales/driver/drivers.json')

In [0]:
df_json.display()

## DDL: Data Definition Language

### Schema defined with StructType() and StructField()

In [0]:
## Easy way to edit a schema - copy the original schema and edit the column type.
my_ddl_schema = ''' 
                Item_Identifier STRING, 
                Item_Weight STRING, 
                Item_Fat_Content STRING, 
                Item_Visibility DOUBLE, 
                Item_Type STRING, 
                Item_MRP DOUBLE, 
                Outlet_Identifier STRING, 
                Outlet_Establishment_Year INT, 
                Outlet_Size STRING, 
                Outlet_Location_Type STRING, 
                Outlet_Type STRING, 
                Item_Outlet_Sales DOUBLE
                '''

In [0]:
data_new = spark.read.format('csv') \
                .schema(my_ddl_schema) \
                .option('header',True)\
                .load("/Volumes/sample_data/sales/bigmart/BigMart Sales.csv")


In [0]:
data_new.printSchema()

### StructType() Schema


In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

In [0]:
my_struct_schema = StructType([
    StructField('Item_Identifier',StringType(),nullable=False),
    StructField('Item_Weight',DoubleType(), nullable=True),
    StructField('Item_Fat_Content',StringType(),nullable=True),
    StructField('Item_Visibility',DoubleType(), nullable=True),
    StructField('Item_MRP',DoubleType(), nullable=True),
    StructField('Outlet_Identifier',StringType(), nullable=False),
    StructField('Outlet_Establishment_Year',IntegerType(), nullable=True),
    StructField('Outlet_Size',StringType(),nullable=True),
    StructField('Outlet_Location_Type',StringType(),nullable=True),
    StructField('Outlet_Type',StringType(),nullable=True),
    StructField('Item_Outlet_Sales',DoubleType(), nullable=True)
    ])

## SELECT and ALIAS

In [0]:
aliases ={
    'Item_Identifier':'item_id',
    'Item_Weight':'item_weight',
    'Item_Fat_Content':'fat_content',
    'Item_Visibility':'visibility',
    'Item_Type': 'item_type',
    'Item_MRP':'mrp',
    'Outlet_Identifier':'outlet_id',
    'Outlet_Establishment_Year':'outlet_est_year',
    'Outlet_Size':'outlet_size',
    'Outlet_Location_Type':'outlet_loc_type',
    'Outlet_Type':'outlet_type',
    'Item_Outlet_Sales':'sales'
}

df_main = data.select([data[col].alias(aliases[col]) for col in data.columns])
df_main.display()


## FILTER

In [0]:
""" 
Filter the data with fat content = Regular
Slice the data with item_type=Soft drink and weight<10
Fetch data with Tier in (Tier 1 or Tier 2) amd outliet size is null
"""
## Scenario 1
df_main.filter(col('fat_content')=='Regular').display()


In [0]:
## Scenario
df_main.filter((col('item_type')=='Soft Drinks') & (col('item_weight')<10)).display()

In [0]:
## Scenario 3
df_main.filter((col('outlet_size').isNull()) & (col('outlet_loc_type').isin(['Tier 1','Tier 2']))).display()

## Rename Column

In [0]:
example = data.withColumnRenamed("Item_Weight","item_wt").display()

## withColumn

In [0]:
## Create a new column
from pyspark.sql.functions import *
"""lit() let you put a fixed value into a dataframe expression
withColumn() allows you to create new column and modify existing column(s)
"""

df = data.withColumn('flag', lit('new'))
df.display()

In [0]:
df.withColumn('multiplier', col("Item_Weight") * col("Item_MRP")).display()

In [0]:
## Modify existing column
df.withColumn('Item_Fat_Content',regexp_replace(col("Item_Fat_Content"), "Low Fat", "LF")) \
    .withColumn('Item_Fat_Content',regexp_replace(col("Item_Fat_Content"), "Regular", "Reg")).display()

## Type Casting

In [0]:
df_test = df.withColumn('Item_Weight', col('Item_Weight').cast(StringType()))
df_test.printSchema()

## Sort and orderBy

In [0]:
df.sort(col('Item_Weight').desc()).display()

In [0]:
df.sort(['Item_Weight','Item_Visibility'],ascending=[False,True]).display()

## Limit

## Drop

In [0]:
df.drop('flag','multiplier').display()

## Drop_Duplicates

In [0]:
df_drop = df.dropDuplicates().display()

## Union and Union ByName